# 18. Supervised Learning: Naive Bayes

## Algorithm Category
**Type**: Supervised Learning - Classification  
**Complexity**: Low  
**Use Case**: Probabilistic classification based on Bayes' theorem

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand Bayes' theorem and the naive independence assumption
- Implement different Naive Bayes variants (Gaussian, Multinomial, Bernoulli)
- Understand when to use each variant
- Handle text classification with Multinomial Naive Bayes
- Apply Naive Bayes to real-world classification problems

## Historical Context

Naive Bayes is based on Bayes' theorem from probability theory:
- Thomas Bayes (1701-1761): Developed Bayes' theorem
- The "naive" assumption (feature independence) was formalized in the 1960s
- Widely used in text classification and spam filtering

**Key Papers/References:**
- Bayes, T. (1763). "An Essay towards solving a Problem in the Doctrine of Chances"
- Domingos, P. & Pazzani, M. (1997). "On the optimality of the simple Bayesian classifier"

## When to Use Naive Bayes

Naive Bayes is appropriate when:
- You need a fast, simple classifier
- Working with text data (email spam, document classification)
- Features are conditionally independent (or approximately so)
- Small training datasets
- Real-time predictions needed
- High-dimensional data (works well with many features)

## Theory & Mechanics

### Mathematical Foundation

Naive Bayes uses Bayes' theorem with the "naive" assumption of feature independence.

**Bayes' Theorem:**
$$P(y|X) = \frac{P(X|y) \cdot P(y)}{P(X)}$$

**Naive Assumption (Independence):**
$$P(X|y) = P(x_1, x_2, ..., x_n|y) = \prod_{i=1}^{n} P(x_i|y)$$

**Classification Rule:**
$$\hat{y} = \arg\max_{y} P(y) \prod_{i=1}^{n} P(x_i|y)$$

### Variants

1. **Gaussian Naive Bayes**: For continuous features
   - Assumes features follow Gaussian distribution
   - $P(x_i|y) = \frac{1}{\sqrt{2\pi\sigma_y^2}} \exp\left(-\frac{(x_i - \mu_y)^2}{2\sigma_y^2}\right)$

2. **Multinomial Naive Bayes**: For discrete counts (e.g., word counts)
   - Uses multinomial distribution
   - $P(x_i|y) = \frac{N_{yi} + \alpha}{N_y + \alpha n}$

3. **Bernoulli Naive Bayes**: For binary features
   - Uses Bernoulli distribution
   - $P(x_i|y) = P(i|y)x_i + (1 - P(i|y))(1 - x_i)$

### How It Works

1. **Training**: Estimate $P(y)$ and $P(x_i|y)$ for each class and feature
2. **Prediction**: Calculate $P(y|X)$ for each class using Bayes' theorem
3. **Decision**: Choose class with highest probability

### Key Hyperparameters

- **alpha (smoothing)**: Additive smoothing parameter (prevents zero probabilities)
- **fit_prior**: Whether to learn class prior probabilities
- **class_prior**: Prior probabilities of classes

### Limitations

- Strong independence assumption (often violated in practice)
- Can be outperformed by more sophisticated methods
- Requires feature independence (or approximate independence)
- May struggle with correlated features


## Implementation

Let's implement different Naive Bayes variants.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    load_iris,  # Iris flower classification dataset
    load_breast_cancer,  # Breast cancer classification dataset
    fetch_20newsgroups  # Text classification dataset (news articles)
)
from sklearn.naive_bayes import (
    GaussianNB,  # Gaussian Naive Bayes (for continuous features)
    MultinomialNB,  # Multinomial Naive Bayes (for discrete counts, e.g., word counts)
    BernoulliNB  # Bernoulli Naive Bayes (for binary features)
)
from sklearn.model_selection import (
    train_test_split,  # Split data into train/test sets
    cross_val_score,  # Cross-validation scoring
    GridSearchCV  # Hyperparameter tuning
)
from sklearn.metrics import (
    accuracy_score,  # Calculate accuracy
    classification_report  # Detailed classification metrics
)
from sklearn.feature_extraction.text import (
    CountVectorizer,  # Convert text to word count vectors
    TfidfVectorizer  # Convert text to TF-IDF vectors (term frequency-inverse document frequency)
)

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.supervised import split_data, evaluate_classifier  # Supervised learning utilities
from src.models.classification import (
    calculate_classification_metrics,  # Calculate precision, recall, F1, etc.
    plot_confusion_matrix  # Visualize confusion matrix
)
from src.utils.benchmarking import benchmark_model_training  # Measure training time
from src.utils.validation import (
    validate_model_output,  # Check if predictions are valid
    check_cross_validation_stability  # Check CV stability
)

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# EXAMPLE 1: Gaussian Naive Bayes (Continuous Features)
# ============================================

# Gaussian Naive Bayes assumes features follow a Gaussian (normal) distribution
# Good for continuous numerical features (like measurements)

# Load Iris dataset (multiclass classification)
iris = load_iris()  # Returns a Bunch object
X = pd.DataFrame(iris.data, columns=iris.feature_names)  # Features: flower measurements
y = pd.Series(iris.target, name='Species')  # Target: flower species

print(f"Dataset Shape: {X.shape}")  # Output: (150, 4) - 150 flowers, 4 features
print(f"Classes: {iris.target_names.tolist()}")  # Output: ['setosa', 'versicolor', 'virginica']

# Split data into training and test sets
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

# ============================================
# TRAINING GAUSSIAN NAIVE BAYES
# ============================================

# Create GaussianNB model
# GaussianNB assumes each feature follows a normal distribution per class
# It learns: mean (μ) and variance (σ²) for each feature in each class
model = GaussianNB()

# Train the model
# During training, it estimates:
# 1. P(y) - Prior probability of each class (how common each class is)
# 2. P(x_i|y) - Probability of feature x_i given class y (using Gaussian distribution)
model.fit(X_train, y_train)

print("\nGaussian Naive Bayes:")
# Class priors: P(y) - probability of each class before seeing features
print(f"Class priors: {model.class_prior_}")  # Example: [0.33, 0.33, 0.33] for balanced classes
print(f"Number of classes: {len(model.classes_)}")  # Number of classes (3)

# ============================================
# MAKING PREDICTIONS
# ============================================

# Make predictions
y_pred = model.predict(X_test)  # Class predictions (0, 1, or 2)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)  # Compare predictions to true labels
print(f"Test Accuracy: {accuracy:.3f}")  # Display accuracy

# ============================================
# PROBABILITY ESTIMATES
# ============================================

# Get probability estimates
y_pred_proba = model.predict_proba(X_test)
print(f"\nSample prediction probabilities:")
print(f"  Sample 0: {y_pred_proba[0]}")
print(f"  Predicted class: {iris.target_names[y_pred[0]]}")


In [ ]:
# ============================================
# EXAMPLE 2: Multinomial Naive Bayes (For Count/Discrete Data)
# ============================================

# Multinomial Naive Bayes is designed for count data (like word counts in text)
# Here we demonstrate it on discretized continuous data
# In practice, MultinomialNB is primarily used for text classification

# Load Breast Cancer dataset (binary classification)
cancer = load_breast_cancer()  # Returns Bunch object
X_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)  # Features: medical measurements
y_cancer = pd.Series(cancer.target, name='Target')  # Target: tumor type (0 = benign, 1 = malignant)

# ============================================
# DISCRETIZING FEATURES: Converting to Counts
# ============================================

# MultinomialNB requires discrete (integer) counts, not continuous values
# We discretize by multiplying by 10 and converting to integers
# This converts continuous measurements to integer counts
X_cancer_discrete = (X_cancer * 10).astype(int)
# Example: 0.5 becomes 5, 1.23 becomes 12
# This is a simplified approach - in practice, use proper binning

# ============================================
# TRAIN/TEST SPLIT
# ============================================

# Split data into training and test sets
X_c_train, X_c_test, y_c_train, y_c_test = split_data(
    X_cancer_discrete, y_cancer, test_size=0.2, random_state=42
)

# ============================================
# TRAINING MULTINOMIAL NAIVE BAYES
# ============================================

# Create MultinomialNB model
# alpha=1.0: Smoothing parameter (Laplace smoothing)
#   - Prevents zero probabilities when a feature value never appears in a class
#   - alpha=1.0 is default (adds 1 to all counts)
model_multi = MultinomialNB(alpha=1.0)

# Train the model
# During training, it estimates:
# 1. P(y) - Prior probability of each class
# 2. P(x_i|y) - Probability of feature value x_i given class y (using multinomial distribution)
model_multi.fit(X_c_train, y_c_train)

# ============================================
# MAKING PREDICTIONS
# ============================================

# Make predictions
y_c_pred = model_multi.predict(X_c_test)  # Class predictions (0 or 1)

# Calculate accuracy
cancer_accuracy = accuracy_score(y_c_test, y_c_pred)  # Compare predictions to true labels

print("Multinomial Naive Bayes (on discretized continuous data):")
print(f"  Test Accuracy: {cancer_accuracy:.3f}")  # Display accuracy
print(f"  Classes: {cancer.target_names.tolist()}")  # Output: ['malignant', 'benign']

# Note: MultinomialNB works better on true count data (like text word counts)
# This example shows it can work on discretized continuous data, but GaussianNB is usually better


## Text Classification Example

Let's use Multinomial Naive Bayes for text classification (its primary use case).


In [ ]:
# ============================================
# TEXT CLASSIFICATION: MultinomialNB's Primary Use Case
# ============================================

# Multinomial Naive Bayes is excellent for text classification
# It works with word counts (how many times each word appears in a document)
# This is the most common use case for MultinomialNB

# Create simple text classification example
# In practice, you'd use real text datasets like 20newsgroups, email spam, etc.
texts = [
    "machine learning is great",  # Tech content
    "python programming language",  # Tech content
    "data science algorithms",  # Tech content
    "deep learning neural networks",  # Tech content
    "buy now cheap price",  # Spam content
    "sale discount offer",  # Spam content
    "limited time deal",  # Spam content
    "special promotion today"  # Spam content
]
labels = [0, 0, 0, 0, 1, 1, 1, 1]  # 0 = tech, 1 = spam
# Binary classification: tech vs spam

# ============================================
# TEXT TO FEATURES: CountVectorizer
# ============================================

# CountVectorizer converts text documents to word count vectors
# Each document becomes a vector where each element is the count of a word
vectorizer = CountVectorizer()
# fit_transform() learns vocabulary from texts and converts to counts
X_text = vectorizer.fit_transform(texts)
# Returns: sparse matrix (8 documents × vocabulary_size)
# Each row is a document, each column is a word, value is word count

y_text = np.array(labels)  # Convert labels to NumPy array

print(f"Text features shape: {X_text.shape}")  # Output: (8, vocabulary_size)
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")  # Number of unique words
print(f"Sample words: {list(vectorizer.vocabulary_.keys())[:10]}")  # First 10 words

# ============================================
# TRAIN/TEST SPLIT
# ============================================

# Split text data into training and test sets
X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    X_text, y_text, test_size=0.25, random_state=42
)
# 75% for training, 25% for testing

# ============================================
# TRAINING MULTINOMIAL NAIVE BAYES ON TEXT
# ============================================

# Create MultinomialNB model for text classification
model_text = MultinomialNB(alpha=1.0)  # alpha=1.0: Laplace smoothing

# Train the model
# During training, it learns:
# 1. P(y) - Prior probability of each class (tech vs spam)
# 2. P(word|y) - Probability of each word given class
#   - Example: P("machine"|tech) might be high
#   - Example: P("buy"|spam) might be high
model_text.fit(X_text_train, y_text_train)

# ============================================
# MAKING PREDICTIONS
# ============================================

# Make predictions on test set
y_text_pred = model_text.predict(X_text_test)  # Class predictions (0 or 1)

# Calculate accuracy
text_accuracy = accuracy_score(y_text_test, y_text_pred)  # Compare predictions to true labels

print(f"\nText Classification Results:")
print(f"  Test Accuracy: {text_accuracy:.3f}")  # Display accuracy
print(f"  Predictions: {y_text_pred}")  # Predicted classes
print(f"  Actual: {y_text_test}")  # True classes

# Interpretation:
# - MultinomialNB works well for text because it models word counts
# - It learns which words are more common in each class
# - Fast and effective for spam detection, document classification, etc.


## Validation & Testing

Let's validate our models and compare different variants.


In [ ]:
# ============================================
# VALIDATION 1: Comparing Different Naive Bayes Variants
# ============================================

# Different Naive Bayes variants are designed for different data types
# We'll test all three to see which works best for the Iris dataset

# Dictionary of Naive Bayes variants to test
variants = {
    'Gaussian': GaussianNB(),  # For continuous features (normal distribution)
    'Multinomial': MultinomialNB(alpha=1.0),  # For count data (discrete)
    'Bernoulli': BernoulliNB(alpha=1.0)  # For binary features
}

# Use Iris dataset for comparison
results = {}  # Store accuracy for each variant

# Test each variant
for name, nb_model in variants.items():
    if name == 'Multinomial':
        # MultinomialNB requires discrete (integer) counts
        # Discretize continuous features by multiplying and converting to int
        X_discrete = (X * 10).astype(int)
        # Use same train/test split indices
        nb_model.fit(X_discrete.iloc[X_train.index], y_train)
        pred = nb_model.predict(X_discrete.iloc[X_test.index])
    elif name == 'Bernoulli':
        # BernoulliNB requires binary features (0 or 1)
        # Binarize by comparing to median (above median = 1, below = 0)
        X_binary = (X > X.median()).astype(int)
        # Use same train/test split indices
        nb_model.fit(X_binary.iloc[X_train.index], y_train)
        pred = nb_model.predict(X_binary.iloc[X_test.index])
    else:
        # GaussianNB works directly with continuous features
        nb_model.fit(X_train, y_train)
        pred = nb_model.predict(X_test)
    
    # Calculate accuracy
    acc = accuracy_score(y_test, pred)  # Compare predictions to true labels
    results[name] = acc  # Store result
    print(f"{name} Naive Bayes: Accuracy = {acc:.3f}")

# ============================================
# FINDING BEST VARIANT
# ============================================

# Find variant with highest accuracy
best_variant = max(results, key=results.get)
# max(..., key=...) finds the key (variant name) with maximum value (accuracy)
print(f"\nBest variant: {best_variant}")

# Interpretation:
# - GaussianNB usually works best for continuous features (like Iris measurements)
# - MultinomialNB and BernoulliNB require data transformation
# - Choose the variant that matches your data type


In [ ]:
# ============================================
# VALIDATION 2: Cross-Validation
# ============================================

# Cross-validation splits data into k folds and tests on each fold
# More reliable than single train/test split

# Perform 5-fold cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
# model: The GaussianNB model
# X, y: All data (will be split internally)
# cv=5: 5 folds (5 train/test splits)
# scoring='accuracy': Use accuracy as evaluation metric
# Returns: array of 5 accuracy scores (one per fold)

# Calculate statistics across folds
cv_mean = cv_scores.mean()  # Average accuracy across all folds
cv_std = cv_scores.std()  # Standard deviation (measure of variability)

print("Cross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")  # Average ± variability
print(f"  Individual fold scores: {cv_scores}")  # Accuracy for each of the 5 folds

# ============================================
# STABILITY CHECK: Is Performance Consistent?
# ============================================

# Check if cross-validation results are stable (low variation)
# Unstable results suggest model is sensitive to data split
stability = check_cross_validation_stability(cv_scores, threshold=0.1)
# threshold=0.1: Variation should be less than 10% of mean
# Returns dictionary with stability analysis

print(f"  Is Stable: {stability['is_stable']}")  # True if variation < threshold

# ============================================
# VALIDATION 3: Checking Model Output Validity
# ============================================

# validate_model_output() checks if predictions are valid
# task_type='classification' tells validator this is classification
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
# Returns dictionary with validation results

# ============================================
# ASSERTIONS: Automated Validation Checks
# ============================================

# Check: Predictions must be valid
assert validation_result['valid'], "Invalid predictions!"
# If predictions are invalid (wrong shape, wrong values, etc.), stop execution

# Check: CV accuracy must be better than random
assert cv_mean > 0.5, "CV accuracy should be better than random!"
# For 3-class classification, random = 1/3 ≈ 0.333

print("\n✓ Validation checks passed")  # All checks passed!


## Real-World Application

Let's tune hyperparameters and visualize class probabilities.


In [ ]:
# Hyperparameter tuning (smoothing parameter)
alphas = [0.1, 0.5, 1.0, 2.0, 5.0]
alpha_scores = []

for alpha in alphas:
    nb = MultinomialNB(alpha=alpha)
    # Use discretized data
    X_discrete = (X * 10).astype(int)
    scores = cross_val_score(nb, X_discrete, y, cv=5, scoring='accuracy')
    alpha_scores.append(scores.mean())
    print(f"Alpha={alpha}: CV Accuracy = {scores.mean():.3f}")

optimal_alpha = alphas[np.argmax(alpha_scores)]
print(f"\nOptimal alpha: {optimal_alpha}")

# Visualize
plt.figure(figsize=(10, 6))
plt.plot(alphas, alpha_scores, 'o-')
plt.axvline(x=optimal_alpha, color='r', linestyle='--', label=f'Optimal α={optimal_alpha}')
plt.xlabel('Smoothing Parameter (α)')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Naive Bayes: Effect of Smoothing Parameter')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize class probabilities
proba_df = pd.DataFrame(y_pred_proba, columns=iris.target_names)
proba_df['predicted'] = [iris.target_names[p] for p in y_pred]
proba_df['actual'] = [iris.target_names[a] for a in y_test.values]

print("Sample Predictions with Probabilities:")
print(proba_df.head(10))

# Plot probability distributions
plt.figure(figsize=(12, 5))
for idx, class_name in enumerate(iris.target_names):
    plt.subplot(1, 3, idx + 1)
    plt.hist(y_pred_proba[y_test.values == idx, idx], bins=20, alpha=0.7, edgecolor='black')
    plt.xlabel('Predicted Probability')
    plt.ylabel('Frequency')
    plt.title(f'Probability Distribution: {class_name}')
    plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Summary & Key Takeaways

### Key Concepts Learned

1. **Naive Bayes Basics**
   - Based on Bayes' theorem with independence assumption
   - Fast training and prediction
   - Provides probability estimates

2. **Variants**
   - **Gaussian**: For continuous features (assumes normal distribution)
   - **Multinomial**: For count data (text classification, word counts)
   - **Bernoulli**: For binary features

3. **Key Parameters**
   - **alpha (smoothing)**: Prevents zero probabilities (Laplace smoothing)
   - **fit_prior**: Whether to learn class priors from data

4. **Best Practices**
   - Use appropriate variant for your data type
   - Tune smoothing parameter (alpha)
   - Works well for text classification
   - Fast baseline for comparison

### When to Use Naive Bayes

✅ **Good for:**
- Text classification (spam detection, sentiment analysis)
- High-dimensional data
- Small training datasets
- Real-time predictions
- When you need probability estimates
- Fast baseline classifier

❌ **Not ideal for:**
- Correlated features (independence assumption violated)
- Complex relationships between features
- When highest accuracy is required
- Very small datasets (may overfit)

### Next Steps

- Try **Complement Naive Bayes** for imbalanced text classification
- Explore **Bayesian Networks** for handling dependencies
- Compare with **Logistic Regression** for similar use cases
- Use for **spam filtering** and **document classification**
